# Lab 05-01: Compare Models, Thinking Levels, and Routing

**Book chapter:** Chapter 05: Models and Building Agents With ADK  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's agents handle millions of requests, from simple intent classification
to multi-rule return decisions. Choosing one model for everything either wastes money
on easy tasks or loses quality on hard ones. In this lab, you measure the latency and
token use of Flash-Lite, Flash, and Pro models, compare thinking levels, build a router
that sends each request to the smallest suitable model, and estimate the cost.

## Learning Objectives

By the end of this lab, you will be able to:

- Measure latency and token use for different Gemini tiers
- Explain how thinking levels trade cost and latency for reasoning
- Build a simple model router
- Estimate the cost of a model choice at production volume

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for about 20 short Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Tier | Strength | Typical use |
|---|---|---|
| **Flash-Lite** | Lowest latency and cost | High-volume classification, routing, extraction |
| **Flash** | Balance of quality, speed, and cost | Most agent conversations and tool use |
| **Pro** | Strongest reasoning | Complex multi-step decisions and planning |

**Routing** sends each request to the smallest model that can handle it, which is one of the
most effective ways to control cost without losing quality.

## Setup

In [ ]:
%pip install --quiet --upgrade google-genai

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Choose the models and define the test tasks

`MODELS` maps each tier to a model ID, and `TASKS` holds three representative tasks:
classification, extraction, and multi-rule reasoning. `ask` sends a request and
returns the answer, the latency, and the token counts from `usage_metadata`.
`safe_ask` reports a model that is not available instead of stopping the lab.

> **Note:** Model IDs change often, and some models are previews. Check the Google
> models page and update `MODELS` before you run the cell.

In [ ]:
import os
import time

from google import genai
from google.genai import types

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
client = genai.Client(vertexai=True, project=PROJECT, location="global")

# Model IDs change often. Check the Google models page and update these.
MODELS = {"Flash-Lite": "gemini-3.5-flash-lite",
          "Flash": os.environ.get("GEMINI_MODEL", "gemini-3.8-flash"),
          "Pro": "gemini-3.1-pro-preview"}

TASKS = {
    "classify": "Classify this message as RETURN, ORDER_STATUS, or OTHER. "
                "Reply with one word: 'Where is my parcel? I ordered boots "
                "last week.'",
    "extract": "Return JSON with order_id, item, and reason for: 'Order "
               "10421: the Trailhead boots are a size too small.'",
    "reason": "Policy: unworn items 30 days, Gold members 60 days, "
              "footwear worn outdoors is not returnable unless defective, "
              "defective items 12 months. A Gold member wore boots on one "
              "hike, and the sole split after 50 days. Can they return "
              "the boots? Answer in two sentences.",
}


def ask(model, prompt, **settings):
    """Sends one request and returns the answer, latency, and token use."""
    start = time.perf_counter()
    response = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(**settings))
    usage = response.usage_metadata
    return {"text": " ".join((response.text or "").split()),
            "seconds": time.perf_counter() - start,
            "input": usage.prompt_token_count or 0,
            "output": usage.candidates_token_count or 0,
            "thinking": usage.thoughts_token_count or 0}


def safe_ask(model, prompt, **settings):
    """Like ask, but reports an unavailable model instead of stopping."""
    try:
        return ask(model, prompt, **settings)
    except Exception as error:
        print(f"  {model} is not available: {str(error)[:60]}")
        return None

## Step 2: Run every task on every model and compare the results

Run every task on every model. Each line shows the latency, input tokens, output tokens, thinking tokens, and the start of the answer.

In [ ]:
results = {}
for task, prompt in TASKS.items():
    print(f"\n=== Task: {task}")
    for tier, model in MODELS.items():
        r = safe_ask(model, prompt)
        if r:
            results[(task, tier)] = r
            print(f"{tier:10} {r['seconds']:5.1f}s  in={r['input']:<4} "
                  f"out={r['output']:<4} thinking={r['thinking']:<5} "
                  f"{r['text'][:45]}")

**What to look for:** all tiers usually handle classification and extraction,
while Pro takes longer and uses more thinking tokens. Quality differences show up
mainly on the reasoning task.

## Step 3: Compare a low and a high thinking level on the reasoning task

Thinking models reason before they answer. The thinking level trades latency and cost for more careful reasoning.

In [ ]:
for level in (types.ThinkingLevel.LOW, types.ThinkingLevel.HIGH):
    r = safe_ask(MODELS["Flash"], TASKS["reason"],
                 thinking_config=types.ThinkingConfig(thinking_level=level))
    if r:
        print(f"\nThinking {level.name}: {r['seconds']:.1f}s, "
              f"{r['thinking']} thinking tokens\n  {r['text']}")

**What to look for:** the high level uses more thinking tokens and more time. Check whether it changes the answer for this task.

## Step 4: Route each request to the smallest model that can handle it

A router is a small, fast model that classifies each request so that expensive models
are used only when needed. If Pro is not available, the request falls back to Flash.

In [ ]:
ROUTER = ("Classify the request as SIMPLE (a lookup, greeting, or single "
          "fact) or COMPLEX (several rules, calculations, or planning). "
          "Reply with one word.")
REQUESTS = ["What time does the Denver store open on Saturdays?",
            TASKS["reason"],
            "Plan a care routine for leather and synthetic boots used on "
            "wet trails, and explain the trade-offs."]
for request in REQUESTS:
    route = safe_ask(MODELS["Flash-Lite"], request,
                     system_instruction=ROUTER)
    complex_request = route and "COMPLEX" in route["text"].upper()
    tier = "Pro" if complex_request else "Flash"
    answer = safe_ask(MODELS[tier], request)
    if answer is None and tier == "Pro":  # Fall back if Pro is unavailable.
        tier, answer = "Flash", safe_ask(MODELS["Flash"], request)
    if answer:
        print(f"\n[{tier}] {request[:60]}\n  {answer['text'][:110]}")

**What to look for:** the store-hours question goes to Flash, and the multi-rule questions go to Pro.

## Step 5: Estimate the cost of the reasoning task on each model

Enter the current prices from the Gemini pricing page in `PRICES` as
`(input_price, output_price)` in USD per million tokens, for example
`"Flash": (0.30, 2.50)`, and run the cell. Thinking tokens are billed as output tokens.

In [ ]:
# Enter USD prices per million (input, output) tokens from the pricing page.
PRICES = {"Flash-Lite": None, "Flash": None, "Pro": None}
for tier in MODELS:
    r, price = results.get(("reason", tier)), PRICES[tier]
    if r and price:
        billed_output = r["output"] + r["thinking"]
        cost = (r["input"] * price[0] + billed_output * price[1]) / 1e6
        print(f"{tier:10} ${cost:.6f} per request, "
              f"${cost * 100_000:,.2f} per 100,000 requests")
    elif r:
        print(f"{tier:10} add prices to PRICES to estimate the cost")

**Try it:** calculate how much routing saves compared with sending every request to Pro.

## Check Your Understanding

**1. Why are thinking tokens important when you estimate cost?**

<details><summary>Show answer</summary>

They are billed as output tokens, so a model or thinking level that reasons more can cost much more than its visible answer suggests.

</details>

**2. When is a router worth its extra call?**

<details><summary>Show answer</summary>

When traffic mixes simple and complex requests. A cheap classification call keeps most requests on a smaller model and reserves the expensive model for hard cases.

</details>

**3. What should an agent do if its preferred model is unavailable?**

<details><summary>Show answer</summary>

Fall back to another suitable model, as the router does, rather than failing the request.

</details>

## Summary

You compared Gemini tiers on latency, tokens, and answers; saw how thinking levels change
cost and latency; built a router; and estimated cost at production volume. This lab created
no resources, so there is nothing to clean up.